
# Homework 24 — train and register model in MLflow

Notebook:
1. Загружаем набор данных о диабете (`scaled=False`)
2. trains пайплайн `StandardScaler + RandomForestRegressor`
3. logs params и метрики  MLflow
4. регистрируем модель **diabets**
5. скачиваем заргеистрированную модель и сохраняем  через fastapi сервис 

In [3]:
import sys, pkg_resources, mlflow
print(sys.executable)
print("pkg_resources ok")
print(mlflow.__version__)

/Users/andrejosadcij/Documents/mlservice_hw24/.venv/bin/python
pkg_resources ok
2.11.3


In [1]:
pip install mlflow

Note: you may need to restart the kernel to use updated packages.


In [4]:

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib
from pathlib import Path


In [5]:

MLFLOW_TRACKING_URI = "http://localhost:8080"
EXPERIMENT_NAME = "diabets-experiment"
REGISTERED_MODEL_NAME = "diabets"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)


In [6]:

db = load_diabetes(scaled=False)
X = db.data
y = db.target
feature_names = list(db.feature_names)

print(feature_names)
print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
(442, 10) (442,)


In [7]:

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(
            n_estimators=200,
            max_depth=8,
            min_samples_split=4,
            random_state=42,
        )),
    ]
)

with mlflow.start_run(run_name="rf_diabets_pipeline") as run:
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_samples_split", 4)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME,
    )

    run_id = run.info.run_id

print("run_id =", run_id)
print("mae =", mae)
print("r2 =", r2)


Successfully registered model 'diabets'.
2026/03/17 21:02:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabets, version 1


run_id = ef46c3c7f1264061bac9de423a1651ca
mae = 43.93924830983982
r2 = 0.44717457052956244


Created version '1' of model 'diabets'.


In [8]:

versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)
model_uri = f"models:/{REGISTERED_MODEL_NAME}/{latest_version}"

print("latest_version =", latest_version)
print("model_uri =", model_uri)


latest_version = 1
model_uri = models:/diabets/1


In [9]:

loaded_model = mlflow.sklearn.load_model(model_uri)

service_model_path = Path("..") / "model" / "diabets_model.pkl"
service_model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(loaded_model, service_model_path)

print("saved:", service_model_path.resolve())


saved: /Users/andrejosadcij/Documents/mlservice_hw24/model/diabets_model.pkl


In [10]:

sample = {
    "age": float(X_test[0][0]),
    "sex": float(X_test[0][1]),
    "bmi": float(X_test[0][2]),
    "bp": float(X_test[0][3]),
    "s1": float(X_test[0][4]),
    "s2": float(X_test[0][5]),
    "s3": float(X_test[0][6]),
    "s4": float(X_test[0][7]),
    "s5": float(X_test[0][8]),
    "s6": float(X_test[0][9]),
}
sample


{'age': 61.0,
 'sex': 1.0,
 'bmi': 25.8,
 'bp': 90.0,
 's1': 280.0,
 's2': 195.4,
 's3': 55.0,
 's4': 5.0,
 's5': 4.9972,
 's6': 90.0}


После выполнения ноутбука:
1. в MLflow появится experiment и registered model **diabets**
2. локально сохранится файл `model/diabets_model.pkl`
3. FastAPI сервис начнёт отвечать реальными предсказаниями
